In [4]:
from __future__ import annotations

import logging

import pandas as pd

from credit_risk.evaluations.evaluations import evaluate_dataset
from credit_risk.evaluations.reporting import _get_evaluation_dir
from credit_risk.utils.config import create_path,read_config

logger = logging.getLogger("v2_temporal_analysis")

In [5]:
from pathlib import Path
import os

if "project_path" not in globals():
    project_path = Path.cwd().parent
    os.chdir(project_path)

print("Project path:", project_path)

Project path: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [6]:
config = read_config(project_path)

In [7]:
from credit_risk.modelling.artifacts import load_model_artifacts, load_training_config

model_config = config

model, preprocessor = load_model_artifacts(model_config)

In [18]:
approach = config["parameters"]["modelling_approach"]
target = config['parameters']['target']['name']
validation_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "validation_df",
    approach,
)

oot_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "oot_df",
    approach,
)

validation_df = pd.read_parquet(validation_path)
oot_df = pd.read_parquet(oot_path)

temporal_df = pd.concat(
    [validation_df, oot_df],
    ignore_index=True,
)

In [9]:
from credit_risk.pipelines.evaluation import evaluate_split

In [10]:
vintage_results = []

for vintage in sorted(temporal_df["vintage"].dropna().unique()):
    vintage_df = temporal_df.loc[temporal_df["vintage"].eq(vintage)].copy()

    evaluation, y_true, y_proba = evaluate_split(
        model=model,
        preprocessor=preprocessor,
        df=vintage_df,
        config=config,
        return_predictions=True,
    )

    metrics = evaluation["ds_metrics"]

    top_k = pd.DataFrame(evaluation["top_k_metrics"])

    top_5 = top_k.loc[top_k["top_fraction"].eq(0.05)].iloc[0]

    top_10 = top_k.loc[top_k["top_fraction"].eq(0.10)].iloc[0]

    vintage_results.append(
        {
            "vintage": int(vintage),
            "population": len(vintage_df),
            "events": int(y_true.sum()),
            "event_rate": float(y_true.mean()),
            "roc_auc": metrics["roc_auc"],
            "pr_auc": metrics["pr_auc"],
            "ks": metrics["ks"],
            "brier_score": metrics["brier_score"],
            "log_loss": metrics["log_loss"],
            "top_5_capture": top_5["event_capture_rate"],
            "top_5_precision": top_5["precision"],
            "top_5_lift": top_5["lift"],
            "top_10_capture": top_10["event_capture_rate"],
            "top_10_precision": top_10["precision"],
            "top_10_lift": top_10["lift"],
        }
    )

temporal_results = (
    pd.DataFrame(vintage_results).sort_values("vintage").reset_index(drop=True)
)

temporal_results

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penal

,vintage,population,events,event_rate,roc_auc,pr_auc,ks,brier_score,log_loss,top_5_capture,top_5_precision,top_5_lift,top_10_capture,top_10_precision,top_10_lift
0,2019,178958,6209,0.034695,0.705465,0.211779,0.296869,0.030919,0.164334,0.280077,0.194345,5.601484,0.355452,0.123324,3.554478
1,2020,183537,1305,0.007110,0.786554,0.242382,0.438799,0.006074,0.035525,0.426054,0.060586,8.520934,0.518008,0.036831,5.179992
2,2021,193330,1030,0.005328,0.738429,0.142251,0.336155,0.004864,0.030120,0.344660,0.036723,6.892847,0.431068,0.022966,4.310680
3,2022,193813,2077,0.010717,0.711666,0.159777,0.303766,0.012867,0.071204,0.334136,0.071613,6.682474,0.394319,0.042256,3.943045


In [11]:
temporal_results[
    [
        "vintage",
        "population",
        "events",
        "event_rate",
    ]
]

,vintage,population,events,event_rate
0,2019,178958,6209,0.034695
1,2020,183537,1305,0.007110
2,2021,193330,1030,0.005328
3,2022,193813,2077,0.010717


In [12]:
temporal_results["event_rate_yoy_change"] = temporal_results["event_rate"].pct_change()

temporal_results[
    [
        "vintage",
        "event_rate",
        "event_rate_yoy_change",
    ]
]

,vintage,event_rate,event_rate_yoy_change
0,2019,0.034695,NaN
1,2020,0.007110,-0.795065
2,2021,0.005328,-0.250708
3,2022,0.010717,1.011480


In [13]:
temporal_results[
    [
        "vintage",
        "roc_auc",
        "pr_auc",
        "ks",
        "top_5_capture",
        "top_5_lift",
        "top_10_capture",
        "top_10_lift",
    ]
]

,vintage,roc_auc,pr_auc,ks,top_5_capture,top_5_lift,top_10_capture,top_10_lift
0,2019,0.705465,0.211779,0.296869,0.280077,5.601484,0.355452,3.554478
1,2020,0.786554,0.242382,0.438799,0.426054,8.520934,0.518008,5.179992
2,2021,0.738429,0.142251,0.336155,0.344660,6.892847,0.431068,4.310680
3,2022,0.711666,0.159777,0.303766,0.334136,6.682474,0.394319,3.943045


In [14]:
temporal_results["pr_auc_to_baseline"] = (
    temporal_results["pr_auc"] / temporal_results["event_rate"]
)

In [15]:
temporal_results[
    [
        "vintage",
        "event_rate",
        "pr_auc",
        "pr_auc_to_baseline",
    ]
]

,vintage,event_rate,pr_auc,pr_auc_to_baseline
0,2019,0.034695,0.211779,6.103965
1,2020,0.007110,0.242382,34.088959
2,2021,0.005328,0.142251,26.700322
3,2022,0.010717,0.159777,14.909383


In [16]:
age_mix = (
    temporal_df.groupby(["vintage", "observation_age"])
    .size()
    .rename("population")
    .reset_index()
)

age_mix["population_share"] = age_mix["population"] / age_mix.groupby("vintage")[
    "population"
].transform("sum")

age_mix

,vintage,observation_age,population,population_share
0,2019,3,49285,0.275400
1,2019,6,47473,0.265275
2,2019,9,43605,0.243661
3,2019,12,38595,0.215665
4,2020,3,49335,0.268801
5,2020,6,47377,0.258133
6,2020,9,44726,0.243689
7,2020,12,42099,0.229376
8,2021,3,49502,0.256049
9,2021,6,48821,0.252527


In [19]:
age_event_rate = (
    temporal_df.groupby(["vintage", "observation_age"])
    .agg(
        population=("observation_age", "size"),
        events=(target, "sum"),
        event_rate=(target, "mean"),
    )
    .reset_index()
)

age_event_rate

,vintage,observation_age,population,events,event_rate
0,2019,3,49285,1687,0.034229
1,2019,6,47473,1854,0.039054
2,2019,9,43605,1538,0.035271
3,2019,12,38595,1130,0.029278
4,2020,3,49335,568,0.011513
5,2020,6,47377,309,0.006522
6,2020,9,44726,246,0.005500
7,2020,12,42099,182,0.004323
8,2021,3,49502,235,0.004747
9,2021,6,48821,265,0.005428


In [20]:
baseline_vintage = temporal_results["vintage"].min()

comparison = temporal_results.copy()

for metric in [
    "roc_auc",
    "pr_auc",
    "ks",
    "top_5_capture",
    "top_5_lift",
]:
    baseline = comparison.loc[
        comparison["vintage"].eq(baseline_vintage),
        metric,
    ].iloc[0]

    comparison[f"{metric}_vs_first"] = comparison[metric] - baseline

comparison

,vintage,population,events,event_rate,roc_auc,pr_auc,ks,brier_score,log_loss,top_5_capture,...,top_10_capture,top_10_precision,top_10_lift,event_rate_yoy_change,pr_auc_to_baseline,roc_auc_vs_first,pr_auc_vs_first,ks_vs_first,top_5_capture_vs_first,top_5_lift_vs_first
0,2019,178958,6209,0.034695,0.705465,0.211779,0.296869,0.030919,0.164334,0.280077,...,0.355452,0.123324,3.554478,NaN,6.103965,0.000000,0.000000,0.000000,0.000000,0.000000
1,2020,183537,1305,0.007110,0.786554,0.242382,0.438799,0.006074,0.035525,0.426054,...,0.518008,0.036831,5.179992,-0.795065,34.088959,0.081089,0.030603,0.141930,0.145976,2.919450
2,2021,193330,1030,0.005328,0.738429,0.142251,0.336155,0.004864,0.030120,0.344660,...,0.431068,0.022966,4.310680,-0.250708,26.700322,0.032964,-0.069528,0.039286,0.064583,1.291364
3,2022,193813,2077,0.010717,0.711666,0.159777,0.303766,0.012867,0.071204,0.334136,...,0.394319,0.042256,3.943045,1.011480,14.909383,0.006201,-0.052002,0.006897,0.054058,1.080991


In [ ]:
print("=== TEMPORAL ANALYSIS CHECKLIST ===")

print(
    "Vintages evaluated:",
    temporal_results["vintage"].tolist(),
)

print(
    "Largest event-rate vintage:",
    temporal_results.loc[
        temporal_results["event_rate"].idxmax(),
        "vintage",
    ],
)

print(
    "Best ROC-AUC vintage:",
    temporal_results.loc[
        temporal_results["roc_auc"].idxmax(),
        "vintage",
    ],
)

print(
    "Worst ROC-AUC vintage:",
    temporal_results.loc[
        temporal_results["roc_auc"].idxmin(),
        "vintage",
    ],
)

print(
    "Best PR-AUC vintage:",
    temporal_results.loc[
        temporal_results["pr_auc"].idxmax(),
        "vintage",
    ],
)

print(
    "Worst PR-AUC vintage:",
    temporal_results.loc[
        temporal_results["pr_auc"].idxmin(),
        "vintage",
    ],
)

=== TEMPORAL ANALYSIS CHECKLIST ===
Vintages evaluated: [2019, 2020, 2021, 2022]
Largest event-rate vintage: 2019
Best ROC-AUC vintage: 2020
Worst ROC-AUC vintage: 2019
Best PR-AUC vintage: 2020
Worst PR-AUC vintage: 2021
